# 06 — Service Layer: activity plan suggestions

Ties everything together: `find_pois` (category/amenity matching, from KG
Modelling) + `GtfsRouter` (travel time, from the Reasoning Layer) into
`service/activity_planner.py`'s `plan_activities()` — which generates actual
multi-stop **itineraries**, not just ranked POI lists or bare travel-time
chains. Matches the one-pager's motivation directly ("someone else planned a
day... for you").

Each POI category has a **default visit duration** (`DEFAULT_VISIT_MINUTES`)
— rough, stated assumptions, not derived from any data source — that count
toward the time budget alongside travel, and that can be overridden per
query. This is what turns "here's a chain of transport links" into "here's an
actual afternoon plan."

Scope, stated plainly (see `service/activity_planner.py`'s docstring):
- Plans are at most 2 stops (origin → stop1 → stop2).
- `time_budget_min` covers travel + both visits, but **not** a return trip
  home — the plan ends when you're done at the last stop.

Two ways to use this notebook: fixed, reproducible example queries
(Sections 1-3) and a live interactive widget (Section 5) — the widget cell
won't show meaningful output when run headlessly via `nbconvert`/export, it
needs an actual running Jupyter kernel with a click.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

from rdflib import Graph
from reasoning.gtfs_routing import GtfsRouter
from service.activity_planner import plan_activities, format_plan, DEFAULT_VISIT_MINUTES
import matplotlib.pyplot as plt

g = Graph()
g.parse("../kg/vienna_mobility_kg.ttl", format="turtle")
router = GtfsRouter(date="20260815")
print(f"KG: {len(g)} triples, GTFS router ready")

# a few well-connected, recognizable named origins -- avoids ever having to
# type raw lon/lat (which is exactly how an earlier session found a real bug:
# (lon, lat) typed in the wrong order looks like valid coordinates but
# silently points somewhere off the map)
LANDMARKS = {
    "Karlsplatz": (16.368948, 48.200955),
    "Stephansplatz": (16.372931, 48.208611),
    "Praterstern": (16.393889, 48.216111),
    "Schwedenplatz": (16.377500, 48.212222),
    "Westbahnhof": (16.337778, 48.196389),
}

## Default visit durations per category

In [ ]:
for cls, minutes in DEFAULT_VISIT_MINUTES.items():
    print(f"  {cls:20} {minutes} min")

## 1. Single-stop itinerary: a dog-friendly park, 60-minute budget

Budget now covers travel *and* the visit, not just travel — a park with a
2.9-minute commute and the default 45-minute visit uses 48 of the 60 minutes.

In [ ]:
origin_lon, origin_lat = LANDMARKS["Karlsplatz"]

plans = plan_activities(g, router, origin_lon, origin_lat,
                         interests=[{"label": "a dog-friendly park", "poi_classes": ["Park"],
                                     "required_amenities": ["Dogs allowed"]}],
                         time_budget_min=60, depart_after="14:00:00")
print(f"{len(plans)} option(s):\n")
for p in plans[:3]:
    print(format_plan(p, "14:00:00"))
    print()

## 2. A real 2-stop itinerary: dog-friendly park, then a library

Default visit times (45 + 30 min) plus travel, all within one budget.

In [ ]:
plans2 = plan_activities(g, router, origin_lon, origin_lat,
                          interests=[
                              {"label": "a dog-friendly park", "poi_classes": ["Park"], "required_amenities": ["Dogs allowed"]},
                              {"label": "a library", "poi_classes": ["Library"]},
                          ],
                          time_budget_min=120, depart_after="14:00:00")
print(f"{len(plans2)} plan(s):\n")
for p in plans2[:3]:
    print(format_plan(p, "14:00:00"))
    print()

best_plan = plans2[0]

## 3. Same itinerary, but with overridden visit times

Say you only want a quick 15-minute stop at the park (versus the 45-minute
default) and a longer 45-minute library visit (versus 30) — pass
`visit_minutes` per interest.

In [ ]:
plans3 = plan_activities(g, router, origin_lon, origin_lat,
                          interests=[
                              {"label": "a dog-friendly park", "poi_classes": ["Park"],
                               "required_amenities": ["Dogs allowed"], "visit_minutes": 15},
                              {"label": "a library", "poi_classes": ["Library"], "visit_minutes": 45},
                          ],
                          time_budget_min=90, depart_after="14:00:00")
print(f"{len(plans3)} plan(s) with custom visit times:\n")
for p in plans3[:2]:
    print(format_plan(p, "14:00:00"))
    print()

## 4. What the best default-visit-time plan looks like on the map

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

stop_lons = router.stops["stop_lon"].to_numpy()
stop_lats = router.stops["stop_lat"].to_numpy()
ax.scatter(stop_lons, stop_lats, s=3, color="lightgray", zorder=1)

points = [(origin_lon, origin_lat, "origin (Karlsplatz)")]
for stop in best_plan["stops"]:
    points.append((stop["lon"], stop["lat"], f"{stop['label']}: {stop['name']} ({stop['visit_min']:.0f} min visit)"))

xs, ys = zip(*[(p[0], p[1]) for p in points])
ax.plot(xs, ys, "--", color="#4C72B0", zorder=2)
for lon, lat, label in points:
    ax.scatter([lon], [lat], s=200, zorder=3)
    ax.annotate(label, (lon, lat), textcoords="offset points", xytext=(8, 8), fontsize=9)

pad = max(0.01, (max(xs) - min(xs)) * 0.6, (max(ys) - min(ys)) * 0.6)
ax.set_xlim(min(xs) - pad, max(xs) + pad)
ax.set_ylim(min(ys) - pad, max(ys) + pad)

ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title(f"Suggested itinerary -- {best_plan['total_itinerary_min']:.0f} min total (travel + visiting)")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 5. Interactive: ask for your own itinerary

Pick a starting landmark, one or two interests, an optional required amenity
for each, a visit-time slider per interest (pre-filled with that category's
default — adjust if you want), and a total time budget. Leaving interest 2 as
"(none)" gives a single-stop suggestion instead of a 2-stop plan.

**Needs a live kernel to actually use** — running this notebook via
`nbconvert --execute` (or any headless export) renders the widgets but can't
simulate a button click, so this cell alone won't show a result outside a
real Jupyter session. Sections 1-3 above already demonstrate the exact same
underlying call, so the logic is validated either way.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

POI_OPTIONS = ["Museum", "Library", "BathingSite", "Park", "SwimmingPool", "PlaygroundArea", "TouristAttraction"]

origin_dd = widgets.Dropdown(options=list(LANDMARKS.keys()), description="Start at:")

interest1_dd = widgets.Dropdown(options=POI_OPTIONS, value="Park", description="Interest 1:")
amenity1_txt = widgets.Text(description="Amenity 1:", placeholder="e.g. Dogs allowed (optional)")
visit1_slider = widgets.IntSlider(value=DEFAULT_VISIT_MINUTES["Park"], min=5, max=180, step=5, description="Visit 1 (min):")

interest2_dd = widgets.Dropdown(options=["(none)"] + POI_OPTIONS, value="Library", description="Interest 2:")
amenity2_txt = widgets.Text(description="Amenity 2:", placeholder="optional")
visit2_slider = widgets.IntSlider(value=DEFAULT_VISIT_MINUTES["Library"], min=5, max=180, step=5, description="Visit 2 (min):")

def _sync_visit1_default(change):
    visit1_slider.value = DEFAULT_VISIT_MINUTES.get(change["new"], 45)
interest1_dd.observe(_sync_visit1_default, names="value")

def _sync_visit2_default(change):
    if change["new"] != "(none)":
        visit2_slider.value = DEFAULT_VISIT_MINUTES.get(change["new"], 45)
interest2_dd.observe(_sync_visit2_default, names="value")

budget_slider = widgets.IntSlider(value=120, min=20, max=360, step=10, description="Budget (min):")
depart_txt = widgets.Text(value="14:00:00", description="Depart after:")
go_button = widgets.Button(description="Find a plan", button_style="primary")
output = widgets.Output()

def on_click(_):
    with output:
        clear_output()
        lon, lat = LANDMARKS[origin_dd.value]
        interests = [{"label": interest1_dd.value, "poi_classes": [interest1_dd.value],
                      "required_amenities": [amenity1_txt.value] if amenity1_txt.value else None,
                      "visit_minutes": visit1_slider.value}]
        if interest2_dd.value != "(none)":
            interests.append({"label": interest2_dd.value, "poi_classes": [interest2_dd.value],
                               "required_amenities": [amenity2_txt.value] if amenity2_txt.value else None,
                               "visit_minutes": visit2_slider.value})
        plans = plan_activities(g, router, lon, lat, interests=interests,
                                 time_budget_min=budget_slider.value, depart_after=depart_txt.value)
        if not plans:
            print("No plan found within that budget -- try a larger budget or a different combination.")
        for p in plans[:3]:
            print(format_plan(p, depart_txt.value))
            print()

go_button.on_click(on_click)

display(widgets.VBox([origin_dd,
                       interest1_dd, amenity1_txt, visit1_slider,
                       interest2_dd, amenity2_txt, visit2_slider,
                       budget_slider, depart_txt, go_button, output]))

## Findings

**Visit time turns a route chain into a plan.** Without it, "budget" only
meant "how much time can I spend on buses/trains," which isn't how anyone
actually thinks about planning an afternoon. Folding visit time into the
budget — and critically, into the *chaining* (leg 2 departs after leg 1's
visit ends, not right after arrival) — is what makes the output a genuine
itinerary with real clock times, not a routing artifact.

**The defaults are honest guesses, not data.** `DEFAULT_VISIT_MINUTES` has no
empirical backing — it's what seemed reasonable per category. Making them
overridable per query (rather than hardcoding them) means the tool is honest
about that uncertainty instead of presenting a guess as a fact.

**Still no return trip modelled.** A plan's "total time" ends when you leave
the last stop, not when you get home. Adding that back is a small, mechanical
extension (one more `travel_time_to()` call from the last stop back to
origin) — not done here to keep the scope of this pass focused, but worth
flagging as the obvious next increment rather than a hidden gap.